In [16]:
import json
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer, util
from scipy.special import rel_entr

# Set up OpenAI API key
API_KEY_PATH = Path('openai_key.txt')

def load_openai_api_key():
    env_key = os.environ.get('OPENAI_API_KEY')
    if env_key:
        return env_key.strip()
    if API_KEY_PATH.exists():
        file_key = API_KEY_PATH.read_text().strip()
        if file_key:
            return file_key
    raise EnvironmentError('Set OPENAI_API_KEY or create openai_key.txt in the notebook directory.')

client = OpenAI(api_key=load_openai_api_key())

# Load Sentence Transformer Model
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

In [2]:
# ==============================================================================
# Core Selection Functions & LLM Generation
# ==============================================================================
def rag_based_relevance(x: str, ad_texts: list): 
    x_embedding = model.encode(x)
    ads_embedding = model.encode(ad_texts)
    bf = (1 + util.dot_score(x_embedding, ads_embedding)[0].numpy()) / 2
    return bf

def randomized_selection(q: np.ndarray, b: np.ndarray):
    g = np.random.gumbel(loc=0, scale=1, size=b.shape[0])
    g = np.exp(g)
    adjusted_bids = q * b * g
    winner = np.argmax(adjusted_bids)
    
    denominator = adjusted_bids[winner] / b[winner]
    adjusted_bids[winner] = 0
    second_winner = np.argmax(adjusted_bids)
    return winner, adjusted_bids[second_winner] / denominator, q * b * g

def multi_ad_greedy_selection(q: np.ndarray, b: np.ndarray, k: int):
    g = np.random.gumbel(loc=0, scale=1, size=b.shape[0])
    g = np.exp(g)
    adjusted_bids = q * b * g
    sorted_order = np.argsort(-adjusted_bids)
    winners = sorted_order[:k]
    k_th_score = adjusted_bids[sorted_order[k]]
    
    payment = []
    for winner in winners:
        payment.append(k_th_score / adjusted_bids[winner] * b[winner])
    return winners, payment, q * b * g


def segment_based_RAG_generation(prompt: str, advertiser: str, ad: str, previous_queries: list, previous_outputs: list):
    """Used for Single Allocation, generated increments based on the previous text"""
    if len(previous_queries) == 0:
        query = f'''{prompt}\n please respond to this question for only one sentence while also advertise {advertiser} with this context >>
        {ad}
        Make sure to connect the answer and the advertisement very naturally, not something like appending the ads after just answering the question.
        Focus on answering the question, there shouldn't be too much advertisment in the output.
        Make sure that response should be one sentence.
        '''
    else:
        query = f'''
        You must continue your answer to my original query.
        Your previous response was
        >> {previous_outputs[-1]}

        And you now should advertise {advertiser}, but without hurting the coherency of the entire document.
        Here's some contexts about {advertiser}
        >>  {ad}

        Make sure that there is one new sentence.
        Write the entire document, which merges your previous response and new paragraph.
        '''
    
    messages = []
    for i in range(len(previous_queries)):
        messages.append({"role": "user", "content": previous_queries[i]})
        messages.append({"role": "assistant", "content": previous_outputs[i]})
    messages.append({"role": "user", "content": query})
    
    response = client.chat.completions.create(
        model = "gpt-4-turbo", 
        logprobs = False,
        temperature = 1.0,
        max_tokens = 300,
        messages=messages)
    
    return query, response.choices[0].message.content

def multi_allocation_generation(prompt: str, advertisers: list, ads: list):
    """Used for Multi Allocation to generate three sentences at one time"""
    query = f'''{prompt}\n please respond to this question for only three sentence while
        (1) advertise {advertisers[0]} with this context >>
        {ads[0]}
        (2) advertise {advertisers[1]} with this context >>
        {ads[1]}
        (3) advertise {advertisers[2]} with this context >>
        {ads[2]}
        
        Make sure to connect the answer and the advertisement very naturally, not something like appending the ads after just answering the question.
        Focus on answering the question, there shouldn't be too much advertisment in the output.
        Make sure to advertise all three brands and ensure that the response is three sentences.
        '''
    
    messages = [{"role": "user", "content": query}]
    response = client.chat.completions.create(
        model = "gpt-4-turbo",
        logprobs = False,
        temperature = 1.0,
        max_tokens = 300,
        messages=messages)
    
    return response.choices[0].message.content

In [3]:
# ==============================================================================
# 1. single_alloc_with_replacement (Updated for Appending)
# ==============================================================================
def run_single_alloc_with_replacement(N_trials, bids, advertisers, ad_texts, prompt, num_of_segments=3, save_path='single_repl_logs.json', dependent=True, use_llm=True):
    save_file = Path(save_path)
    
    # 1. Load existing logs
    if save_file.exists():
        try:
            with open(save_file, 'r', encoding='utf-8') as f:
                all_logs = json.load(f)
            print(f" Found {len(all_logs)} existing logs in {save_path}.")
        except json.JSONDecodeError:
            print(f" Warning: {save_path} corrupted. Starting fresh.")
            all_logs = []
    else:
        all_logs = []

    initial_count = len(all_logs)
    print(f" Appending {N_trials} new trials...")

    # 2. Start loop
    for i in tqdm(range(N_trials), desc="Single With Replacement"):
        current_trial_idx = initial_count + i  
        
        trial_log = []
        v = np.copy(bids)
        previous_queries, previous_outputs = [], []
        
        np.random.seed(current_trial_idx)
        
        if not dependent:
            q = rag_based_relevance(prompt, ad_texts)
        
        for t in range(num_of_segments):
            # Dynamic Update q
            if dependent:
                curr_context = prompt + ('\n' + previous_outputs[-1] if previous_outputs else '')
                q = rag_based_relevance(curr_context, ad_texts)
                
            # Auction Logic
            k, payment, s = randomized_selection(q, bids) 
            
            # LLM Generation
            if use_llm:
                query, output = segment_based_RAG_generation(prompt, advertisers[k], ad_texts[k], previous_queries, previous_outputs)
            else:
                query, output = '', ''
                
            previous_queries.append(query)
            previous_outputs.append(output)
            
            # Probability distributions calculation
            alloc_probs = (q * bids) / np.sum(q * bids)
            q_tilde_probs = q / np.sum(q)
            
            trial_log.append({
                'round': t + 1,
                'winner': advertisers[k],
                'allocation': {adv: float(a) for adv, a in zip(advertisers, alloc_probs)},
                'q_tilde': {adv: float(qt) for adv, qt in zip(advertisers, q_tilde_probs)},
                'payment': float(payment),
                'relevance': float(q[k]),
                'social_welfare': float(q[k] * v[k]),
                'generated_text': output 
            })
            
        all_logs.append(trial_log)
        
        # 3. Save in real-time
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_logs, f, indent=4, ensure_ascii=False)
        
    print(f" Done! Total logs in {save_path}: {len(all_logs)}")
    return all_logs

In [4]:
# ==============================================================================
# 2. single_alloc_without_replacement (Updated for Appending)
# ==============================================================================
def run_single_alloc_without_replacement(N_trials, bids, advertisers, ad_texts, prompt, num_of_segments=3, save_path='single_no_repl_logs.json', dependent=True, use_llm=True):
    save_file = Path(save_path)
    
    # 1. Load existing logs to enable appending
    if save_file.exists():
        try:
            with open(save_file, 'r', encoding='utf-8') as f:
                all_logs = json.load(f)
            print(f" Found {len(all_logs)} existing logs in {save_path}.")
        except json.JSONDecodeError:
            print(f" Warning: {save_path} corrupted. Starting fresh.")
            all_logs = []
    else:
        all_logs = []

    initial_count = len(all_logs)
    print(f" Appending {N_trials} new trials (Without Replacement)...")
    
    for i in tqdm(range(N_trials), desc="Single Without Replacement"):
        current_trial_idx = initial_count + i # Ensure unique seed for appended trials
        
        trial_log = []
        current_bids = np.copy(bids)
        v = np.copy(bids)
        previous_queries, previous_outputs = [], []
        
        # Set seed for reproducibility
        np.random.seed(current_trial_idx)
        
        if not dependent:
            q = rag_based_relevance(prompt, ad_texts)
        
        for t in range(num_of_segments):
            if np.sum(current_bids) == 0:
                break
            
            # Update q dynamically based on previous sentence
            if dependent:
                curr_context = prompt + ('\n' + previous_outputs[-1] if previous_outputs else '')
                q = rag_based_relevance(curr_context, ad_texts)
                
            k, payment, s = randomized_selection(q, current_bids)
            
            # Generate response via LLM
            if use_llm:
                query, output = segment_based_RAG_generation(prompt, advertisers[k], ad_texts[k], previous_queries, previous_outputs)
            else:
                query, output = '', ''
                
            previous_queries.append(query)
            previous_outputs.append(output)
            
            # ================= "Physical Removal" of Dimensions =================
            # 1. Find indices of advertisers still in the pool
            active_indices = np.where(current_bids > 0)[0]
            
            # 2. Extract active advertisers, bids, and relevance scores
            active_advs = [advertisers[idx] for idx in active_indices]
            active_bids = current_bids[active_indices]
            active_q = q[active_indices]
            
            # 3. Calculate probabilities within the reduced dimension
            sum_qb = np.sum(active_q * active_bids)
            active_alloc_probs = (active_q * active_bids) / sum_qb if sum_qb > 0 else np.ones(len(active_indices)) / len(active_indices)
            
            sum_q = np.sum(active_q)
            active_q_tilde_probs = active_q / sum_q if sum_q > 0 else np.ones(len(active_indices)) / len(active_indices)
            
            # 4. Create dictionaries that shrink over rounds (e.g., length 4 -> 3 -> 2)
            alloc_dict = {adv: float(prob) for adv, prob in zip(active_advs, active_alloc_probs)}
            q_tilde_dict = {adv: float(prob) for adv, prob in zip(active_advs, active_q_tilde_probs)}
            # ====================================================================
            
            trial_log.append({
                'round': t + 1,
                'winner': advertisers[k],
                'allocation': alloc_dict,
                'q_tilde': q_tilde_dict,
                'payment': float(payment),
                'relevance': float(q[k]),
                'social_welfare': float(q[k] * v[k]),
                'generated_text': output
            })
            
            # Important: Set current bid to 0 so they are removed in the next round
            current_bids[k] = 0 
            
        all_logs.append(trial_log)
        
        # Save after each trial to prevent data loss
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_logs, f, indent=4, ensure_ascii=False)
        
    print(f" Done! Total logs in {save_path}: {len(all_logs)}")
    return all_logs

In [5]:
# ==============================================================================
# 3. run_multi_alloc_greedy (Updated for Appending)
# ==============================================================================
def run_multi_alloc_greedy(N_trials, bids, advertisers, ad_texts, prompt, num_of_ads=3, save_path='multi_greedy_logs.json', use_llm=True):
    save_file = Path(save_path)
    
    # 1. Load existing logs to enable appending
    if save_file.exists():
        try:
            with open(save_file, 'r', encoding='utf-8') as f:
                all_logs = json.load(f)
            print(f" Found {len(all_logs)} existing logs in {save_path}.")
        except json.JSONDecodeError:
            print(f" Warning: {save_path} corrupted. Starting fresh.")
            all_logs = []
    else:
        all_logs = []

    initial_count = len(all_logs)
    print(f" Appending {N_trials} new trials (Multi-Greedy)...")
    
    for i in tqdm(range(N_trials), desc="Multi Greedy"):
        current_trial_idx = initial_count + i  # Ensure unique seed for appended trials
        
        trial_log = []
        v = np.copy(bids)
        
        # Set seed for reproducibility
        np.random.seed(current_trial_idx)
        
        # Multi-allocation computes relevance once for the whole set
        q = rag_based_relevance(prompt, ad_texts)
        winners, payments, s = multi_ad_greedy_selection(q, bids, num_of_ads)
        
        # Multi-allocation generates the text once for all winners
        if use_llm:
            output = multi_allocation_generation(prompt, [advertisers[idx] for idx in winners], [ad_texts[idx] for idx in winners])
        else:
            output = ''
        
        # Fixed probability logic as per your correction:
        # 1. Allocation probability depends on both relevance and bids
        alloc_probs = (q * bids) / np.sum(q * bids) if np.sum(q * bids) > 0 else np.ones(len(bids))/len(bids)
        # 2. q_tilde (Target distribution) depends purely on relevance
        q_tilde_probs = q / np.sum(q) if np.sum(q) > 0 else np.ones(len(bids))/len(bids)
        
        for t, winner_idx in enumerate(winners):
            trial_log.append({
                'round': t + 1,
                'winner': advertisers[winner_idx],
                'allocation': {adv: float(a) for adv, a in zip(advertisers, alloc_probs)},
                'q_tilde': {adv: float(qt) for adv, qt in zip(advertisers, q_tilde_probs)},
                'payment': float(payments[t]),
                'relevance': float(q[winner_idx]),
                'social_welfare': float(q[winner_idx] * v[winner_idx]),
                'generated_text': output if t == 0 else '[See Round 1]' 
            })
            
        all_logs.append(trial_log)
        
        # Save after each trial to prevent data loss during long API calls
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_logs, f, indent=4, ensure_ascii=False)
        
    print(f" Done! Total Multi-Greedy logs in {save_path}: {len(all_logs)}")
    return all_logs

# Running Experiments

## Senario 1

In [6]:

prompt = '"What can I visit on a trip to Hawaii?"'

advertisers = ['SunWing Airlines', 'TropicStay', 'WanderBite', 'NovaSkin', 'GridPower Bank']
ads = [
    "Take off with SunWing Airlines, your trusted partner for seamless travel across the Pacific and beyond. With spacious cabins, award-winning in-flight service, and direct routes to the world's most breathtaking destinations, SunWing makes every journey as memorable as the destination itself. Book today and enjoy exclusive deals, flexible cancellation, and a loyalty rewards program that turns miles into memories. Your next adventure starts here — fly with SunWing.",
    "Discover the perfect retreat with TropicStay, the premier platform for booking vacation rentals in the world's most beautiful destinations. From oceanfront villas to cozy jungle hideaways, TropicStay connects travelers with handpicked, locally-owned properties that offer authentic experiences you won't find anywhere else. With instant booking, 24/7 guest support, and verified reviews from real travelers, planning your dream getaway has never been easier. Explore, book, and belong — only with TropicStay.",
    "Unlock the culinary soul of every destination with WanderBite, the app that connects food lovers to the best local restaurants, street food spots, and hidden dining gems around the globe. Whether you're craving fresh poke bowls on the North Shore or a Michelin-starred tasting menu in the city, WanderBite guides you there with curated recommendations, real-time reservations, and exclusive foodie deals. Because the best travel memories are made at the table. Download WanderBite and taste the world.",
    "Introducing NovaSkin, the dermatologist-approved skincare line engineered for the modern lifestyle. From SPF-50 daily moisturizers to overnight repair serums, NovaSkin's lightweight, reef-safe formulas protect and restore your skin whether you're under the office lights or the open sun. Trusted by over 10 million customers worldwide, NovaSkin combines cutting-edge biotechnology with clean, sustainable ingredients to deliver visible results you can feel confident about. Because great skin doesn't take a vacation — and neither does NovaSkin.",
    "Stay charged through every adventure with GridPower Bank, the ultra-slim, high-capacity portable charger built for life on the move. Featuring rapid-charge technology, dual USB-C ports, and a rugged waterproof design, GridPower Bank keeps your devices powered through long flights, beach days, and everything in between. Compact enough to fit in your pocket, powerful enough to charge your laptop twice over — GridPower Bank is the travel essential you didn't know you needed. Power up and go.",
]

bids = np.array([3, 3, 2, 2, 1])

print(f"Experimental Prompt: {prompt}")
print(f"Advertisers: {advertisers}")
print(f"Initialized {len(ads)} matched ads.")

Experimental Prompt: "What can I visit on a trip to Hawaii?"
Advertisers: ['SunWing Airlines', 'TropicStay', 'WanderBite', 'NovaSkin', 'GridPower Bank']
Initialized 5 matched ads.


In [ ]:
num_trials=20

for _ in range(5):
    try:
        # Run the three methods
        run_single_alloc_with_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_with_replacement_logs_scenario_1.json')
        run_single_alloc_without_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_without_replacement_logs_scenario_1.json')
        run_multi_alloc_greedy(num_trials, bids, advertisers, ads, prompt, save_path='Seg_multi_logs_scenario_1.json')
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Senario 2

In [8]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids = np.array([3, 3, 2, 2])

print(f"Experimental Prompt: {prompt}")
print(f"Advertisers: {advertisers}")
print(f"Initialized {len(ads)} matched ads.")

Experimental Prompt: "Can you suggest some books similar to "To Kill a Mockingbird"?"
Advertisers: ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
Initialized 4 matched ads.


In [ ]:
num_trials=20

for _ in range(5):
    try:
        # Run the three methods
        run_single_alloc_with_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_with_replacement_logs_scenario_2.json')
        run_single_alloc_without_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_without_replacement_logs_scenario_2.json')
        run_multi_alloc_greedy(num_trials, bids, advertisers, ads, prompt, save_path='Seg_multi_logs_scenario_2.json')
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Senario 3

In [10]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids = np.array([2, 1, 3, 3])

print(f"Experimental Prompt: {prompt}")
print(f"Advertisers: {advertisers}")
print(f"Initialized {len(ads)} matched ads.")

Experimental Prompt: "Can you suggest some books similar to "To Kill a Mockingbird"?"
Advertisers: ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
Initialized 4 matched ads.


In [ ]:
num_trials=20

for _ in range(5):
    try:
        # Run the three methods
        run_single_alloc_with_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_with_replacement_logs_scenario_3.json')
        run_single_alloc_without_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_without_replacement_logs_scenario_3.json')
        run_multi_alloc_greedy(num_trials, bids, advertisers, ads, prompt, save_path='Seg_multi_logs_scenario_3.json')
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Senario 4

In [13]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
    "Discover the power of connection with SocialHub, the world's leading social media platform. With over two billion active users, SocialHub is your gateway to staying in touch with friends and family, discovering new communities, and sharing your life's moments. Our innovative features, from personalized news feeds to interactive groups, make it easy to engage with what matters most to you. Whether you're promoting your business, staying updated on the latest news, or simply keeping up with loved ones, SocialHub is the ultimate tool to enhance your digital experience. Join us today and be part of a global network where connections come to life!",
    "Experience the refreshing taste of ColaBubbles, the world's favorite soft drink. With its unique blend of flavors and effervescent bubbles, ColaBubbles has been bringing joy to people of all ages for over a century. Whether you're enjoying a moment of relaxation, celebrating with friends, or on the go, ColaBubbles is the perfect companion to quench your thirst and uplift your spirits. Our commitment to quality and tradition ensures every sip is as delightful as the first. Indulge in the classic taste of ColaBubbles and make every moment special. Taste the feeling!",
    "Unleash the bold taste of FizzyPop, the iconic soft drink that invigorates and refreshes like no other. Known for its crisp, refreshing flavor and vibrant effervescence, FizzyPop is the perfect choice for those who dare to live life to the fullest. Whether you're at a party, watching a game, or simply taking a break, FizzyPop brings a burst of excitement to any occasion. With a heritage of quality and a commitment to innovation, every sip of FizzyPop delivers an unmatched experience. Embrace the bold, and make every moment extraordinary with the unmistakable taste of FizzyPop.",
    "Explore the skies with SkyTech, the world's leading aerospace company renowned for its innovation, quality, and reliability. SkyTech designs, manufactures, and services commercial airplanes, defense systems, and space technologies, making global connectivity and exploration possible. Whether you're traveling for business or leisure, SkyTech's state-of-the-art aircraft ensure a safe, comfortable, and efficient journey. With a legacy of pioneering advancements and a commitment to excellence, SkyTech continues to shape the future of aviation. Choose SkyTech and experience the pinnacle of aerospace engineering and performance. Fly with confidence, fly with SkyTech.",
    "Experience the future of aviation with AeroDynamics, the global leader in aerospace innovation and excellence. AeroDynamics designs and manufactures the world’s most advanced commercial aircraft, providing unparalleled comfort, efficiency, and reliability. From cutting-edge technology to sustainable solutions, AeroDynamics is dedicated to shaping the future of air travel. Whether you're embarking on a long-haul journey or a short domestic flight, AeroDynamics ensures a superior flying experience with spacious cabins, innovative features, and top-notch safety standards. Trust AeroDynamics for a seamless and enjoyable journey every time. Fly smarter, fly with AeroDynamics.",
    "Immerse yourself in the world of music with MusicStream, the ultimate destination for streaming your favorite tunes anytime, anywhere. With a vast library of millions of songs, playlists curated just for you, and personalized recommendations, MusicStream puts the power of music discovery in your hands. Whether you're in the mood for chart-topping hits, underground gems, or soothing melodies, MusicStream has something for everyone. Plus, with offline listening capabilities and seamless integration across devices, you can take your music with you wherever you go. Join the millions of music lovers worldwide and unlock endless possibilities with MusicStream. Discover, stream, and experience the joy of music like never before.", 
    "Experience the cutting-edge innovation of BrainChips, the global leader in semiconductor technology. BrainChips' groundbreaking processors power the devices that fuel our modern world, from laptops and desktops to servers and cloud computing systems. With a legacy of pushing the boundaries of technology, BrainChips continues to deliver industry-leading performance, reliability, and security. Whether you're a professional tackling complex tasks or a gamer seeking immersive experiences, BrainChips processors provide the power and efficiency you need. Trust BrainChips to deliver the performance you demand and the reliability you can count on. Join the millions who rely on BrainChips technology and unlock new possibilities for productivity, creativity, and entertainment.",
]


bids = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

print(f"Experimental Prompt: {prompt}")
print(f"Advertisers: {advertisers}")
print(f"Initialized {len(ads)} matched ads.")

Experimental Prompt: "Can you suggest some books similar to "To Kill a Mockingbird"?"
Advertisers: ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']
Initialized 11 matched ads.


In [ ]:
num_trials=100

for _ in range(1):
    try:
        # Run the three methods
        run_single_alloc_with_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_with_replacement_logs_scenario_4.json')
        run_single_alloc_without_replacement(num_trials, bids, advertisers, ads, prompt,save_path='Seg_single_without_replacement_logs_scenario_4.json')
        run_multi_alloc_greedy(num_trials, bids, advertisers, ads, prompt, save_path='Seg_multi_logs_scenario_4.json')
    except Exception as e:
        print(f"Error: {e}, retrying...")